# Jour 1b — Modèles de référence : k-mer + ML classique, et one-hot + CNN

Deux façons de transformer une chaîne d'ADN en quelque chose qu'un modèle peut utiliser,
avant même de toucher à un modèle de langage pré-entraîné :

1. **Fréquences de k-mers** : compter la fréquence de chaque sous-chaîne de longueur k
   (par ex. les 256 4-mers possibles), normaliser -> un vecteur de taille fixe -> ML
   classique (régression logistique).
2. **One-hot + CNN** : encoder chaque nucléotide en un vecteur one-hot de dimension 4
   (A/C/G/T), empiler en un tenseur (4, 200), et laisser un petit CNN 1D apprendre
   lui-même ses propres caractéristiques directement à partir de la séquence brute.

In [ ]:
import sys
sys.path.append("src")

import numpy as np
import torch
from data import load_all
from featurize import kmer_matrix, one_hot_batch
from models.baselines import make_kmer_classifier, OneHotCNN
from eval import evaluate_sklearn, evaluate_logits, count_params, measure_latency_sklearn, measure_latency_torch

splits = load_all("../../data/supervised/processed")
train, val = splits["train"], splits["val"]

## Modèle de référence 1 : fréquences de k-mers + régression logistique

**À faire :** complétez le code ci-dessous pour calculer les matrices de k-mers (k=4) sur
train/val, entraîner un classifieur logistique (`make_kmer_classifier("logreg")`), puis
l'évaluer sur validation avec `evaluate_sklearn`.

In [ ]:
K = 4
# TODO : calculez les matrices de k-mer pour train et val avec kmer_matrix(..., k=K)
X_train_kmer = ...
X_val_kmer = ...
y_train, y_val = train["label"].to_numpy(), val["label"].to_numpy()

# TODO : créez un classifieur avec make_kmer_classifier("logreg") et entraînez-le (fit)
kmer_clf = ...

# TODO : évaluez-le sur validation avec evaluate_sklearn(...)
kmer_metrics = ...
print("k-mer + logreg:", kmer_metrics)

## Modèle de référence 2 : nucléotides one-hot + petit CNN

**À faire :** encodez les fenêtres en one-hot (`one_hot_batch`, longueur 200), instanciez
`OneHotCNN`, puis entraînez-le pendant 5 époques (Adam, lr=1e-3,
`binary_cross_entropy_with_logits`, mini-lots de 256).

In [ ]:
WINDOW = 200
# TODO : encodez les fenêtres d'entraînement et de validation en one-hot, convertissez en tensors torch
X_train_oh = ...
X_val_oh = ...
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

# TODO : instanciez le modèle OneHotCNN(seq_len=WINDOW) et un optimiseur Adam (lr=1e-3)
cnn = ...
optimizer = ...

n = X_train_oh.shape[0]
for epoch in range(5):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        # TODO : calculez les logits du CNN sur ce mini-lot, puis la perte BCE-with-logits
        logits = ...
        loss = ...
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    print(f"epoch {epoch+1}: loss={epoch_loss/n:.4f}")

In [ ]:
# TODO : passez X_val_oh dans le CNN (sans gradient) puis évaluez avec evaluate_logits
with torch.no_grad():
    val_logits = ...
cnn_metrics = ...
print("one-hot CNN:", cnn_metrics)

## Comparaison

In [ ]:
import pandas as pd

# TODO : complétez le nombre de paramètres et la latence pour chaque modèle
# (count_params / measure_latency_sklearn pour le k-mer, count_params / measure_latency_torch pour le CNN)
comparison = pd.DataFrame([
    {"model": "kmer+logreg", **kmer_metrics,
     "params": ...,
     "latency_ms": ...},
    {"model": "onehot+CNN", **cnn_metrics,
     "params": ...,
     "latency_ms": ...},
])
comparison

### Point de contrôle

Vous devriez avoir un tableau accuracy/F1/params/latence pour les deux modèles de
référence. Gardez ces chiffres (ou le DataFrame `comparison`) — ils serviront pour le
graphique d'efficacité du Jour 4.

Suite : `02_evo2_embeddings_and_classifier.ipynb` — un modèle de fondation génomique
fait-il mieux ?

*Bloqué ? La version complète est dans `solution/01_kmer_and_cnn_baselines.ipynb`.*